# Sleep-EDF: AnySleep — длина контекста147–155

Один запуск: smoke → полный val трёх AnySleep → фиксация выбора по val B → test всех успешных вариантов.
147:core256/halo128;148:core512/halo256;149:целый непрерывный сегмент.
150–152:фиксированный B;153–155:диагностическая подстановка в113 с прежними heads.
019/099/113 читаются из Input, повторного baseline inference нет. Обучения нет.
При GPU OOM сначала уменьшается только batch; если batch1 не помещается, вариант явно пропускается.
Длина контекста и float32 не меняются. Для полного запуска достаточно SMOKE_ONLY=False, TEST_POLICY="all".


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/revlze/sleep-edf-classification.git"
CODE_REF = (
    "701dab6201de71481d45d751e11e802e285d31e7"  # Проверенная версия кода для resume.
)
PROJECT_ROOT = Path("/tmp/sleep-edf-anysleep-context-code-v1")
OUTPUT_ROOT = Path("/kaggle/working/anysleep-context-results")
INPUT_ROOT = (
    None  # Автопоиск sleep-edf-anysleep-context-inputs.zip / распакованного bundle.
)
RESUME_ROOT = (
    None  # Необязательно: ZIP или каталог результатов ЭТОЙ серии и версии кода.
)

SMOKE_ONLY = True  # False: автоматический smoke -> полный val -> выбор по val -> test всех вариантов.
SMOKE_EPOCHS = 160
TEST_POLICY = (
    "all"  # all: все успешные варианты и их смеси получают test probabilities/CSV.
)
DEVICE = "cuda:0"  # Одна T4; память двух GPU не объединяется.
THREADS = max(1, os.cpu_count() or 2)
MAX_BATCH = 4  # Только одинаковые длины; при OOM уменьшается batch, размер контекста сохраняется.


def checkout_code():
    if PROJECT_ROOT.exists():
        current_ref = subprocess.check_output(
            ["git", "config", "sleepedf.bootstrapRef"], cwd=PROJECT_ROOT, text=True
        ).strip()
        if current_ref != CODE_REF:
            raise RuntimeError(
                "Для другого CODE_REF задайте новый PROJECT_ROOT в /tmp."
            )
        print(
            "Используется существующий checkout; для обновления ветки задайте новый PROJECT_ROOT."
        )
        return
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory(prefix="sleep-git-") as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            'print("x-access-token" if "Username" in sys.argv[1] else os.environ["SLEEP_GITHUB_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        env = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "SLEEP_GITHUB_TOKEN": token,
        }
        subprocess.run(
            ["git", "init", "--quiet", "--initial-branch=bootstrap", str(PROJECT_ROOT)],
            check=True,
        )
        subprocess.run(
            ["git", "remote", "add", "origin", REPO_URL], cwd=PROJECT_ROOT, check=True
        )
        subprocess.run(
            [
                "git",
                "-c",
                "credential.helper=",
                "fetch",
                "--quiet",
                "--depth",
                "1",
                "origin",
                CODE_REF,
            ],
            cwd=PROJECT_ROOT,
            env=env,
            check=True,
        )
        subprocess.run(
            ["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"],
            cwd=PROJECT_ROOT,
            check=True,
        )
        subprocess.run(
            ["git", "config", "sleepedf.bootstrapRef", CODE_REF],
            cwd=PROJECT_ROOT,
            check=True,
        )
    del token


checkout_code()
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
print("Код:", commit)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook import reset_commands, run_command  # noqa: E402

reset_commands()

# Устанавливаем только инструмент uv; зависимости проекта устанавливает uv.lock.

UV_VERSION = "0.12.15"
if shutil.which("uv") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", f"uv=={UV_VERSION}"], check=True
    )
UV = shutil.which("uv")
ENV = {
    "UV_PROJECT_ENVIRONMENT": "/tmp/sleep-edf-context-venv",
    "UV_CACHE_DIR": "/tmp/sleep-pretrained-uv-cache",
    "UV_PYTHON_INSTALL_DIR": "/tmp/sleep-pretrained-python",
    "PYTHONPATH": str(PROJECT_ROOT),
    "MPLBACKEND": "Agg",  # Backend только для дочернего CLI, не Kaggle-ядра.
}
UV_PROJECT = PROJECT_ROOT / "environments/pretrained"
print("Ядро notebook:", sys.version)
subprocess.run(["nvidia-smi"], check=True)
run_command(
    [
        UV,
        "sync",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "--python",
        "3.12.12",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)
BASE = [
    UV,
    "run",
    "--project",
    str(UV_PROJECT),
    "--locked",
    "--no-dev",
    "python",
    "-m",
    "src.context_run",
]
ENV.update(
    {
        k: str(THREADS)
        for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
    }
)
run_command(
    [
        UV,
        "run",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "python",
        "-c",
        "from src.pretrained import hardware; import json; print(json.dumps(hardware("
        + repr(DEVICE)
        + "),indent=2))",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)

In [ ]:
import hashlib
from zipfile import ZipFile

import pandas as pd
from IPython.display import display


def unpack(path):
    path = Path(path)
    if path.is_dir():
        return path
    key = hashlib.sha256(
        (str(path) + str(path.stat().st_size) + str(path.stat().st_mtime_ns)).encode()
    ).hexdigest()[:16]
    target = Path("/tmp/sleep-edf-context-inputs") / key
    if (target / ".unpacked").exists():
        return target
    target.mkdir(parents=True, exist_ok=True)
    with ZipFile(path) as z:
        for info in z.infolist():
            dest = (target / info.filename).resolve()
            if (
                not dest.is_relative_to(target.resolve())
                or (info.external_attr >> 16) & 0o170000 == 0o120000
            ):
                raise ValueError("Небезопасный путь в ZIP")
            z.extract(info, target)
    (target / ".unpacked").touch()
    return target


roots = [
    p.parent
    for p in Path("/kaggle/input").rglob("MANIFEST.json")
    if (p.parent / "labels.csv").is_file() and (p.parent / "epochs/val").is_dir()
]
if len(roots) != 1:
    raise RuntimeError(f"Нужен один исходный full_psg_v1; найдено: {roots}")
DATA_ROOT = roots[0]
if INPUT_ROOT is None:
    found = [
        p.parent
        for p in Path("/kaggle/input").rglob("bundle.json")
        if json.loads(p.read_text()).get("kind") == "anysleep_context_inputs_v1"
    ]
    if not found:
        for archive in Path("/kaggle/input").rglob("*.zip"):
            with ZipFile(archive) as z:
                if (
                    "bundle.json" in z.namelist()
                    and json.loads(z.read("bundle.json")).get("kind")
                    == "anysleep_context_inputs_v1"
                ):
                    found.append(unpack(archive))
    if len(found) != 1:
        raise RuntimeError(
            "Подключите один новый sleep-edf-anysleep-context-inputs.zip или укажите INPUT_ROOT явно"
        )
    INPUT_ROOT = found[0]
INPUT_ROOT = unpack(INPUT_ROOT)
if RESUME_ROOT is not None:
    previous = unpack(RESUME_ROOT)
    for path in previous.rglob("*"):
        if (
            not path.is_file()
            or path.name == ".unpacked"
            or (path.name.endswith(".lock") and path.name != "uv.lock")
        ):
            continue
        target = OUTPUT_ROOT / path.relative_to(previous)
        if target.exists():
            if (
                hashlib.sha256(target.read_bytes()).digest()
                != hashlib.sha256(path.read_bytes()).digest()
            ):
                raise ValueError(
                    f"Resume conflict; сохранён существующий файл: {target}"
                )
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)
COMMON = [
    "--data-root",
    str(DATA_ROOT),
    "--input-root",
    str(INPUT_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--device",
    DEVICE,
    "--threads",
    str(THREADS),
    "--max-batch",
    str(MAX_BATCH),
    "--test-policy",
    TEST_POLICY,
]
print("DATA_ROOT:", DATA_ROOT, "\nINPUT_ROOT:", INPUT_ROOT)
print(
    "Сигналы читаются один раз на запись; preprocessing общий для трёх вариантов и трёх checkpoints."
)
print(
    "Обучения нет. Старые probabilities019/082/085/099/102/106/113 и официальные AnySleep weights уже в bundle."
)

In [ ]:
from src.notebook import reset_commands

reset_commands()
run_command(
    [*BASE, *COMMON, "--smoke", "--smoke-epochs", str(SMOKE_EPOCHS)],
    cwd=PROJECT_ROOT,
    env=ENV,
)
if not SMOKE_ONLY:
    run_command([*BASE, *COMMON, "--require-smoke"], cwd=PROJECT_ROOT, env=ENV)

namespace = OUTPUT_ROOT / "smoke" if SMOKE_ONLY else OUTPUT_ROOT
selection = namespace / "selection.json"
if selection.exists():
    print(
        "Выбор только по val ансамбля B:", json.loads(selection.read_text())["selected"]
    )
rows = []
for n in range(147, 156):
    folder = namespace / f"{n:03d}_pretrained"
    metrics = folder / "metrics.json"
    row = {"ID": folder.name}
    if metrics.exists():
        m = json.loads(metrics.read_text())
        row.update(
            status=m["status"],
            val_macro_f1=m["validation"]["macro_f1"],
            test=(folder / "test_predictions.npz").exists(),
            diagnostic_only=m["diagnostic_only"],
            forward_s=m["inference_seconds"],
        )
    for split in ("val", "test"):
        status = folder / f"{split}_status.json"
        if status.exists():
            info = json.loads(status.read_text())
            if info["status"] != "complete":
                row[split + "_issue"] = info.get("error", info["status"])
    rows.append(row)
display(pd.DataFrame(rows))
report = namespace / "context_report.json"
if report.exists():
    r = json.loads(report.read_text())
    print(
        "Ошибки/пропуски:",
        len(r["failures"]),
        "· Все запрошенные test завершены:",
        r["all_requested_test_completed"],
    )
print(
    "153–155 — только диагностическая подстановка с прежними102/106; это НЕ заново обученная113."
)

Последняя ячейка доступна и после Stop. ZIP накопительный: smoke/full, record predictions для resume,
probabilities, метрики и сравнения, все готовые CSV, журнал, конфигурации и код с uv.lock.
Сигналы, embeddings и веса в ZIP результатов не входят; новый Input bundle подключается отдельно.
Resume: тот же CODE_REF, исходный Input bundle и RESUME_ROOT с ZIP результатов.


In [ ]:
from datetime import UTC, datetime

from IPython.display import FileLink, display

from src.notebook import reset_commands

reset_commands()
stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
archive = Path("/kaggle/working") / f"sleep-edf-anysleep-context-results-{stamp}.zip"
run_command(
    [*BASE, "--output-root", str(OUTPUT_ROOT), "--export", str(archive)],
    cwd=PROJECT_ROOT,
    env=ENV,
)
display(
    FileLink(
        str(archive.relative_to(Path.cwd())),
        result_html_prefix="Скачать все результаты и сабмишны одним ZIP: ",
    )
)